# Inference efficiency

Forward-only inference with synthetic inputs already on the selected device.
No training or checkpoint download is required for this architecture benchmark.
Results are saved after each model/batch combination in a timestamped directory.
Charts contain only axes and model labels; error bars span repeat-average timings.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import platform
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import pandas as pd
import torch
from IPython.display import Image, display
from models import get_classification_model
from benchmark import benchmark_inference
from utils.io import save_inference_results
from visualization import plot_inference_results

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
BATCH_SIZES = (1, 32)
IMAGE_SIZE = (224, 224)
NUM_ITERATIONS = 100
WARMUP_ITERATIONS = 50
REPEATS = 5
PRETRAINED = False  # Architecture timing only; no training checkpoint.
NUM_CLASSES = 37
DTYPE = torch.float32
SEED = 42
MODEL_NAMES = {
    "ConvNeXt Tiny": ("convnext_tiny", "convnext_tiny"),
    "Swin Transformer Tiny": ("swin_tiny", "swin_tiny_patch4_window7_224"),
    "ResNet-50": ("resnet50", "resnet50"),
}
OUTPUT_DIR = repo_root / "results" / "inference_efficiency" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
print(f"Device: {DEVICE} | Results: {OUTPUT_DIR}")


In [ ]:
torch.manual_seed(SEED)
# Use classification's names for common fields. Model names/IDs are stored
# per result below; batch size, dtype and timing counts come from the benchmark.
metadata = {
    "dataset": "synthetic_random_inputs",
    "image_size": IMAGE_SIZE,
    "initialization": "pretrained_backbone_fresh_classifier" if PRETRAINED else "random_weights",
    "num_classes": NUM_CLASSES,
    "hardware": torch.cuda.get_device_name() if DEVICE == "cuda" else platform.processor() or platform.machine(),
    "versions": {name: version(name) for name in ("torch", "torchvision", "timm")},
    "cpu_threads": torch.get_num_threads(),
    "seed": SEED,
    "timing": "synchronized wall time; forward only; transfers excluded; median of repeat means",
    "flops": "PyTorch FlopCounterMode; one forward batch; multiply-add=2; unregistered operators may be omitted",
}
results = []
for model_name, (model_id, timm_name) in MODEL_NAMES.items():
    model = get_classification_model(timm_name, pretrained=PRETRAINED, num_classes=NUM_CLASSES).to(dtype=DTYPE)
    for batch_size in BATCH_SIZES:
        inputs = torch.randn(batch_size, 3, *IMAGE_SIZE, device=DEVICE, dtype=DTYPE)
        result = benchmark_inference(model, inputs, DEVICE, NUM_ITERATIONS, WARMUP_ITERATIONS, REPEATS)
        results.append({"model_name": model_name, "model_id": model_id, "timm_model": timm_name, **result})
        saved = save_inference_results(results, OUTPUT_DIR, metadata)
        print(f"{model_name}, batch {batch_size}: {result['batch_latency_ms']:.3f} ms/batch, "
              f"{result['throughput_images_per_second']:.1f} images/s")
        del inputs
    model.to("cpu")
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE == "mps":
        torch.mps.empty_cache()


In [ ]:
display(pd.read_csv(OUTPUT_DIR / "inference_summary.csv"))
for path in plot_inference_results(results, OUTPUT_DIR):
    print(path.name)
    display(Image(filename=str(path)))
print("Saved:", saved)
